In [ ]:
!pip install torch-geometric -q
!pip install torch


In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.explain import Explainer, GNNExplainer

import matplotlib.pyplot as plt

In [ ]:
cols = [
'Severity','Start_Time','End_Time',
'Start_Lat','Start_Lng','End_Lat','End_Lng',
'Distance(mi)',
'Temperature(F)','Humidity(%)','Pressure(in)',
'Visibility(mi)','Wind_Speed(mph)','Precipitation(in)',
'Weather_Condition',
'Junction','Traffic_Signal','Crossing','Stop'
]

df = pd.read_csv('/kaggle/input/datasets/sobhanmoosavi/us-accidents/US_Accidents_March23.csv',
usecols=cols
)

In [ ]:
df = df.sample(50000, random_state=42)
df.reset_index(drop=True, inplace=True)

In [ ]:
df['Start_Time'] = pd.to_datetime(df['Start_Time'], format='mixed')
df['End_Time'] = pd.to_datetime(df['End_Time'], format='mixed')


In [ ]:
df['Hour'] = df['Start_Time'].dt.hour
df['DayOfWeek'] = df['Start_Time'].dt.dayofweek
df['Month'] = df['Start_Time'].dt.month

df['Duration'] = (
    df['End_Time'] - df['Start_Time']
).dt.total_seconds() / 60

df = df.drop(columns=['Start_Time','End_Time'])


In [ ]:
num_cols = df.select_dtypes(include=['float64','int64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include='object').columns

for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

In [ ]:
df = pd.get_dummies(df, columns=['Weather_Condition'])


In [ ]:
y = df['Severity'] - 1
X = df.drop(columns=['Severity'])

feature_names = X.columns


In [ ]:
X = X.astype('float32')


In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)


In [ ]:
X_tensor = torch.tensor(X, dtype=torch.float)
y_tensor = torch.tensor(y.values, dtype=torch.long)

In [ ]:
coords = df[['Start_Lat','Start_Lng']].values

nbrs = NearestNeighbors(n_neighbors=6)
nbrs.fit(coords)

distances, indices = nbrs.kneighbors(coords)

edges = []

for i in range(len(indices)):
    for j in indices[i]:
        edges.append([i,j])

edge_index = torch.tensor(edges).t().contiguous()

In [ ]:
data = Data(
    x=X_tensor,
    edge_index=edge_index,
    y=y_tensor
)


In [ ]:
n = data.num_nodes

train_mask = torch.rand(n) < 0.8
test_mask = ~train_mask

data.train_mask = train_mask
data.test_mask = test_mask

In [ ]:
class GCN(torch.nn.Module):

    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels,64)
        self.conv2 = GCNConv(64,32)
        self.conv3 = GCNConv(32,4)

    def forward(self,x,edge_index):

        x = self.conv1(x,edge_index)
        x = F.relu(x)

        x = self.conv2(x,edge_index)
        x = F.relu(x)

        x = self.conv3(x,edge_index)

        return x

In [ ]:
model = GCN(data.num_features)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [ ]:
for epoch in range(30):

    model.train()

    optimizer.zero_grad()

    out = model(data.x,data.edge_index)

    loss = F.cross_entropy(
        out[data.train_mask],
        data.y[data.train_mask]
    )

    loss.backward()

    optimizer.step()

    if epoch % 5 == 0:
        print("Epoch:",epoch,"Loss:",loss.item())

In [ ]:
model.eval()

pred = model(data.x,data.edge_index).argmax(dim=1)

correct = (
pred[data.test_mask] ==
data.y[data.test_mask]
).sum()

acc = int(correct) / int(data.test_mask.sum())

print("Test Accuracy:",acc)

In [ ]:
explainer = Explainer(
    model=model,
    algorithm=GNNExplainer(epochs=100),
    explanation_type="model",
    node_mask_type="attributes",
    edge_mask_type="object",
    model_config=dict(
        mode="multiclass_classification",
        task_level="node",
        return_type="raw",
    ),
)


In [ ]:
node_index = 100

explanation = explainer(
    x=data.x,
    edge_index=data.edge_index,
    index=node_index
)


In [ ]:
feature_importance = explanation.node_mask.mean(dim=0).detach().cpu()

plt.figure(figsize=(10,5))
plt.bar(range(len(feature_importance)), feature_importance)

plt.title("Feature Importance for Accident Severity")
plt.xlabel("Feature Index")
plt.ylabel("Importance")

plt.show()

In [ ]:
top_idx = torch.argsort(feature_importance, descending=True)[:10]

top_features = [feature_names[int(i)] for i in top_idx]

top_scores = [feature_importance[int(i)].item() for i in top_idx]

plt.figure(figsize=(10,6))

plt.barh(top_features, top_scores)

plt.gca().invert_yaxis()

plt.title("Top Accident Risk Factors (GNNExplainer)")

plt.show()

In [ ]:
top_idx = torch.argsort(feature_importance, descending=True)[:10]

print("Top Risk Factors:\n")

for i in top_idx:
    
    idx = int(i)
    
    print(
        feature_names[idx],
        " : ",
        float(feature_importance[idx])
    )